# Module 07 — Walk-Forward Out-of-Sample Backtest

Final 30% out-of-sample walk-forward backtest. Formation parameters and the 40 structurally eligible pairs are frozen from the training sample.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.backtest import run_walk_forward_backtest, backtest_summary

pd.set_option("display.max_columns", 100)


## 1. Load frozen formation data


In [ ]:
TRAIN_FILE = "data/processed/train_prices.parquet"
TEST_FILE = "data/processed/test_prices.parquet"
ELIGIBLE_FILE = "data/processed/eligible_pairs.parquet"
COINTEGRATED_FILE = "data/processed/cointegrated_pairs.parquet"
RF_FILE = "data/processed/risk_free_rates.parquet"

train_prices = pd.read_parquet(TRAIN_FILE)
test_prices = pd.read_parquet(TEST_FILE)
eligible_pairs = pd.read_parquet(ELIGIBLE_FILE)
cointegrated_pairs = pd.read_parquet(COINTEGRATED_FILE)

assert isinstance(train_prices.index, pd.DatetimeIndex), "train_prices index is not DatetimeIndex."
assert isinstance(test_prices.index, pd.DatetimeIndex), "test_prices index is not DatetimeIndex."
assert train_prices.index.max() < test_prices.index.min(), "Train/test periods overlap."

print("Train:", train_prices.index.min(), "->", train_prices.index.max(), train_prices.shape)
print("Test :", test_prices.index.min(), "->", test_prices.index.max(), test_prices.shape)
print("Eligible pairs:", len(eligible_pairs))


## 2. Historical risk-free rate


In [ ]:
rf_data = pd.read_parquet(RF_FILE)

if isinstance(rf_data, pd.DataFrame):
    if rf_data.shape[1] != 1:
        raise ValueError("risk_free_rates.parquet must contain exactly one rate column.")
    risk_free_rates = rf_data.iloc[:, 0]
else:
    risk_free_rates = pd.Series(rf_data)

risk_free_rates.index = pd.to_datetime(risk_free_rates.index)
risk_free_rates = risk_free_rates.sort_index().astype(float)

if risk_free_rates.abs().median() > 1:
    raise ValueError("Risk-free rates appear to be percentages. Divide them by 100 first.")

risk_free_rates.tail()


## 3. Final backtest configuration


In [ ]:
INITIAL_CAPITAL = 100_000.0
ENTRY_Z = 1.5
TARGET_PROBABILITY = 0.70
MEMORY_WINDOW = 60
MAX_HORIZON_DAYS = 126
N_PATHS = 5000
EWMA_LAMBDA = 0.94
SEED = 42


## 4. Run the walk-forward simulation


In [ ]:
results = run_walk_forward_backtest(
    train_prices=train_prices,
    test_prices=test_prices,
    eligible_pairs=eligible_pairs,
    cointegrated_pairs=cointegrated_pairs,
    risk_free_rates=risk_free_rates,
    initial_capital=INITIAL_CAPITAL,
    entry_z=ENTRY_Z,
    target_probability=TARGET_PROBABILITY,
    memory_window=MEMORY_WINDOW,
    max_horizon_days=MAX_HORIZON_DAYS,
    n_paths=N_PATHS,
    ewma_lambda=EWMA_LAMBDA,
    seed=SEED,
)

trades = results["trades"]
equity_curve = results["equity_curve"]
skipped_signals = results["skipped_signals"]

print("Completed trades:", len(trades))
print("Skipped records :", len(skipped_signals))
print("Final equity    :", equity_curve["equity"].iloc[-1])


## 5. Results and sanity checks


In [ ]:
summary = backtest_summary(
    trades=trades,
    equity_curve=equity_curve,
    initial_capital=INITIAL_CAPITAL,
)
summary


In [ ]:
if not trades.empty:
    display(
        trades[[
            "pair", "entry_date", "exit_date", "entry_z",
            "convergence_horizon_trading_days", "option_calendar_dte",
            "dependent_contracts", "independent_contracts",
            "entry_premium", "exit_value", "pnl",
            "trade_return", "exit_reason",
        ]].head(20)
    )

if not skipped_signals.empty and "reason" in skipped_signals.columns:
    display(skipped_signals["reason"].value_counts())


In [ ]:
if not trades.empty:
    print("Largest single entry premium:", trades["entry_premium"].max())
    print("Median entry premium:", trades["entry_premium"].median())

print("Maximum concurrent positions:", equity_curve["n_open_positions"].max())
print("Minimum cash balance:", equity_curve["cash"].min())


## 6. Save Module 07 outputs


In [ ]:
trades.to_parquet("data/processed/trades.parquet", index=False)
equity_curve.to_parquet("data/processed/equity_curve.parquet", index=True)
skipped_signals.to_parquet("data/processed/skipped_signals.parquet", index=False)

print("Saved Module 07 outputs.")
